In [1]:
import re

In [ ]:
import pandas as pd


excel_path = r"C:\ai_x\lecNote\Project1-Deep-Dish\Deep-Dish\레서피\병합_레시피_항목그대로병합.xlsx"
df = pd.read_excel(excel_path, sheet_name='조리순서')
print(df.head(1))

    항목      제육볶음    매운제육볶음    두부제육볶음       된장찌개  김치찌개    청국장찌개  콩나물무침  시금치나물  \
0  1단계  돼지고기얇게썰기  돼지고기얇게썰기  돼지고기얇게썰기  다시마멸치육수내기  김치볶기  청국장풀어주기  콩나물삶기  시금치삶기   

   도라지무침  ...  브런치뷔페    비건버거   두부스테이크       템페    비건케이크 두유아이스크림  닭가슴살샐러드  \
0  도라지찢기  ...  브런치기획  콩패티 준비  두부 물기제거  템페 슬라이스  건조재료 섞기  두유 데우기  닭가슴살 삶기   

     퀴노아볼    프로틴볼   그릭요거트  
0  퀴노아 삶기  견과류 볶기  요거트 준비  

[1 rows x 381 columns]


In [5]:
# 메뉴별 조리 단계 전처리 
step_labels = df['항목'] # 제외할 내용
menu_steps = {}   # 메뉴별 조리 단계 딕셔너리

for col in df.columns[1:]:
    menu_name = col.strip()
    steps = []

    for step_text in df[col]:
        if pd.isna(step_text):
            continue
        step_clean = str(step_text).strip()
        if step_clean in ['-', ''] or '완성' in step_clean:
            break
        steps.append(step_clean)

    menu_steps[menu_name] = {
        'cooking_steps': f"{len(steps)}단계",
        'cooking_info': '\n'.join(steps)
    }

# 예시 출력
print(menu_steps)

{'제육볶음': {'cooking_steps': '7단계', 'cooking_info': '돼지고기얇게썰기\n양념장만들기\n고기양념재우기\n팬에기름두르기\n고기볶기\n양파넣기\n대파넣기'}, '매운제육볶음': {'cooking_steps': '7단계', 'cooking_info': '돼지고기얇게썰기\n양념장만들기\n고기양념재우기\n팬에기름두르기\n고기볶기\n양파넣기\n대파넣기'}, '두부제육볶음': {'cooking_steps': '7단계', 'cooking_info': '돼지고기얇게썰기\n양념장만들기\n고기양념재우기\n팬에기름두르기\n두부추가하기\n고기볶기\n양파대파넣기'}, '된장찌개': {'cooking_steps': '6단계', 'cooking_info': '다시마멸치육수내기\n두부애호박썰기\n된장풀어넣기\n끓여주기\n간맞추기\n대파넣기'}, '김치찌개': {'cooking_steps': '7단계', 'cooking_info': '김치볶기\n돼지고기볶기\n김치넣고볶기\n육수붓기\n두부넣기\n끓이기\n간맞추기'}, '청국장찌개': {'cooking_steps': '6단계', 'cooking_info': '청국장풀어주기\n육수붓기\n두부넣기\n콩나물넣기\n대파넣기\n간맞추기'}, '콩나물무침': {'cooking_steps': '4단계', 'cooking_info': '콩나물삶기\n찬물에헹구기\n양념무침\n참기름넣기'}, '시금치나물': {'cooking_steps': '4단계', 'cooking_info': '시금치삶기\n찬물에헹구기\n양념무침\n참기름넣기'}, '도라지무침': {'cooking_steps': '4단계', 'cooking_info': '도라지찢기\n소금물에담그기\n양념무침\n참기름넣기'}, '계란말이': {'cooking_steps': '4단계', 'cooking_info': '계란풀기\n팬에기름두르기\n계란말기\n돌돌말기'}, '계란찜': {'cooking_steps': '4단계', 'cooking_info': '계란풀기\n뚝배기에기름바

In [6]:
# 특징맛 전처리 
df_flavor = pd.read_excel(excel_path, sheet_name="특징맛")

# index 기준 (A열)
df_flavor.set_index(df_flavor.columns[0], inplace=True)

# 메뉴 리스트 가져오기 (열 기준)
menu_names = df_flavor.columns.tolist()

# 결과 저장
flavor_info_dict = {}

for menu in menu_names:
    info_lines = []
    for attr, value in df_flavor[menu].items():
        if pd.isna(value) or str(value).strip() == "-":
            continue
        info_lines.append(f"{attr.strip()} : {str(value).strip()}")
    flavor_info_dict[menu] = "\n".join(info_lines)

# 확인 예시
print(f"✅ 제육볶음 flavor_info:\n{flavor_info_dict.get('제육볶음')}")


✅ 제육볶음 flavor_info:
주요맛 : 매콤달콤
매운정도 : 2단계
특징 : 한국대표볶음
영양효과 : 단백질보충
추천상황 : 매운음식좋아할때


In [7]:
# 영양정보 시트 전처리 
df_nut = pd.read_excel(excel_path, sheet_name='영양정보')
# print(df_nut.head(1))
# index 기준 (A열)
df_nut.set_index(df_nut.columns[0], inplace=True)

# 메뉴 리스트 가져오기 (열 기준)
menu_names = df_nut.columns.tolist()

# 결과 저장용 dict
nutrition_info_dict = {}

# 전처리 수행
for menu in menu_names:
    info_lines = []
    for attr, value in df_nut[menu].items():
        if pd.isna(value) or str(value).strip() == "-":
            continue
        info_lines.append(f"{attr.strip()} : {str(value).strip()}")
    nutrition_info_dict[menu] = "\n".join(info_lines)

# 예시 출력
print(f"✅ 제육볶음 nutrition_info:\n{nutrition_info_dict.get('제육볶음')}")


✅ 제육볶음 nutrition_info:
단백질(g) : 25
탄수화물(g) : 12
지방(g) : 18
나트륨(mg) : 1200
식이섬유(g) : 3
비타민C(mg) : 20
칼슘(mg) : 50
철분(mg) : 2
비타민A(μg) : 100


In [8]:
# 서빙 조리팁 데이터 불러오기 
df_surving = pd.read_excel(excel_path, sheet_name='서빙조리팁')
# print(df_surving.head(1))

# index 기준 (A열)
df_surving.set_index(df_surving.columns[0], inplace=True)

# 메뉴 리스트 가져오기 (열 기준)
menu_names = df_surving.columns.tolist()

# 결과 저장용 dict
surviving_info_dict = {}

# 전처리 수행
for menu in menu_names:
    info_lines = []
    for attr, value in df_surving[menu].items():
        if pd.isna(value) or str(value).strip() == "-":
            continue
        info_lines.append(f"{attr.strip()} : {str(value).strip()}")
    surviving_info_dict[menu] = "\n".join(info_lines)

# 예시 출력
print(f"✅ 제육볶음 surving_info:\n{surviving_info_dict.get('제육볶음')}")


✅ 제육볶음 surving_info:
조리팁 : 고기얇게썰기
서빙팁 : 뜨거울때바로
보관팁 : 냉장밀폐보관
영양팁 : 돼지고기비타민B1
건강팁 : 밥과함께드세요


In [9]:
df_last = pd.read_excel(excel_path, sheet_name='조리정보')
# print(df_last.head(1))

# 인덱스 설정 (A열 : 항목명)
df_last.set_index(df_last.columns[0], inplace=True)

# 메뉴 리스트 가져오기 (A열 : 항목명)
menu_list = df_last.columns.tolist()

# 결과 저장용 dict 
cooking_time_dict = {}
difficulty_dict = {}

# 난이도 매핑
difficulty_map = {
    '초급': 'easy',
    '중급': 'medium',
    '고급': 'hard'
}
for menu in menu_list:
    # --- 조리시간 ---
    cooking_time = df_last[menu].iloc[0]
    if isinstance(cooking_time, str) and cooking_time.strip() != '-' and pd.notna(cooking_time):
        match = re.search(r"\d+", cooking_time)
        cooking_time_dict[menu] = match.group() if match else None
    else:
        cooking_time_dict[menu] = None  # ✅ None으로 설정

    # --- 난이도 ---
    difficulty = df_last[menu].iloc[1]
    if isinstance(difficulty, str) and difficulty.strip() != '-' and pd.notna(difficulty):
        difficulty_dict[menu] = difficulty_map.get(difficulty.strip(), None)
    else:
        difficulty_dict[menu] = None  # ✅ None으로 설정

# 확인
print(f"무한리필  조리시간: {cooking_time_dict.get('무한리필')}")
print(f"무한리필  난이도: {difficulty_dict.get('무한리필')}")

무한리필  조리시간: None
무한리필  난이도: None


In [10]:
# menu_name → cooking_steps, cooking_info 정리 / 데이터 프레임으로 변환
processed = pd.DataFrame([
    {'menu_name': k, 
	'cooking_steps': v['cooking_steps'], 
	'cooking_info': v['cooking_info'],
	'flavor_characteristics' : flavor_info_dict.get(k.strip(),None),
	'nutrition_info' : nutrition_info_dict.get(k.strip(),None),
	'serving_tips' : surviving_info_dict.get(k.strip(),None),
	'cooking_time_minutes' : cooking_time_dict.get(k.strip(),None),
	'difficulty_level': difficulty_dict.get(k.strip(), None) 
	}
    for k, v in menu_steps.items()
])
print(processed.head(1))

  menu_name cooking_steps                                       cooking_info  \
0      제육볶음           7단계  돼지고기얇게썰기\n양념장만들기\n고기양념재우기\n팬에기름두르기\n고기볶기\n양파넣기...   

                              flavor_characteristics  \
0  주요맛 : 매콤달콤\n매운정도 : 2단계\n특징 : 한국대표볶음\n영양효과 : 단백...   

                                      nutrition_info  \
0  단백질(g) : 25\n탄수화물(g) : 12\n지방(g) : 18\n나트륨(mg)...   

                                        serving_tips cooking_time_minutes  \
0  조리팁 : 고기얇게썰기\n서빙팁 : 뜨거울때바로\n보관팁 : 냉장밀폐보관\n영양팁 ...                   25   

  difficulty_level  
0           medium  


In [ ]:
# mysql 연결 , menu_id 매핑 
import mysql.connector

DB_CONFIG = mysql.connector.connect(
	host = '210.121.189.12',
	user = 'remoteuser',
	password = 'deepdish0000',
	database = 'deep_dish',
	charset = 'utf8mb4',
	port=3306
) 
cursor = DB_CONFIG.cursor()

# menu_name -> menu_id 조회 
cursor.execute("SELECT menu_name, menu_id FROM menus")
menu_map = dict(cursor.fetchall())
# print(menu_map)  메뉴별 id 조회 완료

# menu_id -> 매핑 추가 
processed['menu_id'] = processed['menu_name'].map(lambda x: menu_map.get(x.strip()))


# 필요한 컬럼만 남기기
final_df = processed[[
	'menu_id', 'cooking_steps', 'cooking_info',
	'flavor_characteristics','nutrition_info',
	'serving_tips','cooking_time_minutes','difficulty_level'
	]]

print(final_df.head(1))

   menu_id cooking_steps                                       cooking_info  \
0        1           7단계  돼지고기얇게썰기\n양념장만들기\n고기양념재우기\n팬에기름두르기\n고기볶기\n양파넣기...   

                              flavor_characteristics  \
0  주요맛 : 매콤달콤\n매운정도 : 2단계\n특징 : 한국대표볶음\n영양효과 : 단백...   

                                      nutrition_info  \
0  단백질(g) : 25\n탄수화물(g) : 12\n지방(g) : 18\n나트륨(mg)...   

                                        serving_tips cooking_time_minutes  \
0  조리팁 : 고기얇게썰기\n서빙팁 : 뜨거울때바로\n보관팁 : 냉장밀폐보관\n영양팁 ...                   25   

  difficulty_level  
0           medium  


In [15]:
## mysql에 정리된 자료 추가 
for _, row in final_df.iterrows():
	if pd.isna(row['menu_id']):
		print(f"menu_id 정보가 없습니다. {row['menu_name']}")
		continue
	try:
		cursor.execute("""
									INSERT INTO menu_recipes(
									menu_id,  
									cooking_steps, 
									cooking_info, 
									flavor_characteristics,
									nutrition_info,
									serving_tips,
									cooking_time_minutes,
									difficulty_level)
									VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
									""", (
										int(row['menu_id']),
												row['cooking_steps'],
												row['cooking_info'],
												row['flavor_characteristics'],
												row['nutrition_info'],
												row['serving_tips'],
												row['cooking_time_minutes'],
												row['difficulty_level']
										)
								)
		# MySQL에 저장
		DB_CONFIG.commit()
	except Exception as e:
		print(f"에러가 발생했습니다. {e}")
# cursor.close()
# DB_CONFIG.close()


In [16]:
cursor.close()
DB_CONFIG.close()

In [17]:
final_df['difficulty_level'].unique()

array(['medium', 'easy', None, 'hard'], dtype=object)